## Import Libraries

In [ ]:
# Installing the libraries with the specified version.
!pip install numpy>=2.3 pandas==2.2.2 matplotlib==3.9 seaborn==0.13.2 -q --user

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Executive Plotly Theme

## Synthetic Dataset Generator##

In [ ]:
#import numpy as np
#import pandas as pd

def generate_synthetic_dataset(
    num_samples=1000,
    features_config=None,
    random_seed=None
):
    """
    Generates a synthetic dataset with multiple columns (features),
    each with its own distribution, data type, noise, and missing rate.

    Parameters
    ----------
    num_samples : int, optional
        Number of data points (rows) to generate.
    features_config : list of dict, optional
        A list of feature configuration dictionaries.
        Each dictionary can have the following keys:
          - name (str): The column name.
          - distribution (str): Distribution type (e.g. 'normal', 'uniform', 'exponential', etc.).
          - dist_params (dict): Dictionary of parameters for the chosen distribution.
                               Defaults to None or empty dict if not provided.
          - data_type (str): 'int', 'float', or 'string' (basic type conversion).
                             For more sophisticated data types, implement custom logic.
          - noise_std (float): Standard deviation of Gaussian noise to add (defaults to 0.0 for no noise).
          - missing_rate (float): Fraction of data in the column to randomly replace with NaN.
          - random_seed (int): Optionally override the main random seed for this column specifically.
    random_seed : int, optional
        Seed for the random number generator, to make results reproducible.

    Returns
    -------
    pd.DataFrame
        A DataFrame containing all requested features (columns).
    """
    if random_seed is not None:
        np.random.seed(random_seed)

    if features_config is None or not isinstance(features_config, list):
        raise ValueError("features_config must be a list of feature configurations.")

    df = pd.DataFrame()

    for feature in features_config:
        col_name = feature.get('name', 'feature')
        distribution = feature.get('distribution', 'normal')
        dist_params = feature.get('dist_params', {})
        data_type = feature.get('data_type', 'float')
        noise_std = feature.get('noise_std', 0.0)
        missing_rate = feature.get('missing_rate', 0.0)

        # If the user provided a separate random seed for this column, override
        feature_seed = feature.get('random_seed', None)
        if feature_seed is not None:
            np.random.seed(feature_seed)

        # Generate base data
        col_data = _generate_base_distribution(
            distribution=distribution,
            dist_params=dist_params,
            num_samples=num_samples
        )

        # Revert seed if needed
        if feature_seed is not None and random_seed is not None:
            np.random.seed(random_seed)

        # Add noise
        if noise_std > 0.0:
            noise = np.random.normal(loc=0.0, scale=noise_std, size=num_samples)
            col_data += noise

        # Convert to requested data type
        col_data = _convert_to_type(col_data, data_type)

        col_data = np.round(col_data).astype(float)

        # Generate missing values
        if missing_rate > 0.0:
            num_missing = int(np.floor(missing_rate * num_samples))
            missing_indices = np.random.choice(num_samples, size=num_missing, replace=False)
            col_data[missing_indices] = np.nan

        # Create a Series and add to DataFrame
        df[col_name] = col_data

    return df

def _generate_base_distribution(distribution, dist_params, num_samples):
    """
    Generates base distribution data as a NumPy array.
    Supported distributions can be expanded as needed.
    """
    distribution = distribution.lower()
    if distribution == 'normal':
        # default loc=0, scale=1 if not specified
        loc = dist_params.get('loc', 0.0)
        scale = dist_params.get('scale', 1.0)
        return np.random.normal(loc=loc, scale=scale, size=num_samples)
    elif distribution == 'uniform':
        # default low=0, high=1 if not specified
        low = dist_params.get('low', 0.0)
        high = dist_params.get('high', 1.0)
        return np.random.uniform(low=low, high=high, size=num_samples)
    elif distribution == 'exponential':
        # default scale=1
        scale = dist_params.get('scale', 1.0)
        return np.random.exponential(scale=scale, size=num_samples)
    elif distribution == 'logistic':
        # default loc=0, scale=1
        loc = dist_params.get('loc', 0.0)
        scale = dist_params.get('scale', 1.0)
        return np.random.logistic(loc=loc, scale=scale, size=num_samples)
    elif distribution == 'beta':
        # default a=2, b=5
        a = dist_params.get('a', 2.0)
        b = dist_params.get('b', 5.0)
        return np.random.beta(a, b, size=num_samples)
    # Add more distributions here if needed
    else:
        raise ValueError(f"Unsupported distribution: {distribution}")


def _convert_to_type(data_array, data_type):
    """
    Converts a NumPy array to the specified data type.
    Basic version supporting int, float, and string.
    Handles NaN for integer types by converting to float64.
    """
    data_type = data_type.lower()
    if data_type == 'float':
        return data_array.astype(float)
    elif data_type == 'int':
        # If there are NaNs, convert to float64 to accommodate them
        if np.isnan(data_array).any():
            return data_array.astype(np.float64)  # Change to float64 to hold NaN
        else:
            return np.round(data_array).astype(int)
    elif data_type == 'string':
        return data_array.astype(str)
    else:
        raise ValueError(f"Unsupported data_type: {data_type}")


if __name__ == "__main__":
    # Example usage: generate 5,000 rows with 3 different features
    features_config = [
        {
            'name': 'module',
            'distribution': 'normal',
            'dist_params': {'loc': 170, 'scale': 10},
            'data_type': 'float',
            'noise_std': 2.0,
            'missing_rate': 0.05
        },
        {
            'name': 'permissions',
            'distribution': 'normal',
            'dist_params': {'loc': 40, 'scale': 12},
            'data_type': 'int',
            'noise_std': 5.0,
            'missing_rate': 0.1
        },
        {
            'name': 'groups',
            'distribution': 'uniform',
            'dist_params': {'low': 1, 'high': 10},
            'data_type': 'int',
            'missing_rate': 0.0
        }
    ]

    df_synthetic = generate_synthetic_dataset(
        num_samples=500000,
        features_config=features_config,
        random_seed=42
    )

    print(df_synthetic.head(15))
    print(df_synthetic.describe(include='all'))


## Load, standardize, combine, and scale the data
* Final generates additional synthetic rows as needed to reach **more than 500,000 records**


In [ ]:
import os
import numpy as np
import pandas as pd

# -----------------------------------------------------------
# File names expected in the working folder / Colab session
# -----------------------------------------------------------
AUTO_FILE = "financial_loan.csv"
VEHICLE_FILE_1 = "automobile_loan_default_1.csv"
VEHICLE_FILE_2 = "automobile_loan_default_2.csv"

# Optional Colab upload helper
try:
    from google.colab import files  # type: ignore
    if not all(os.path.exists(f) for f in [AUTO_FILE, VEHICLE_FILE_1, VEHICLE_FILE_2]):
        print("Upload the three CSV files when prompted.")
        uploaded = files.upload()
except Exception:
    pass

# -----------------------------------------------------------
# Load files
# -----------------------------------------------------------
df_auto = pd.read_csv(AUTO_FILE)
df_vehicle_1 = pd.read_csv(VEHICLE_FILE_1)
df_vehicle_2 = pd.read_csv(VEHICLE_FILE_2)

print("Original file shapes:")
print(f"  financial_loan.csv: {df_auto.shape}")
print(f"  automobile_loan_default_1.csv: {df_vehicle_1.shape}")
print(f"  automobile_loan_default_2.csv: {df_vehicle_2.shape}")


In [ ]:
# Concatenate the two vehicle-loan-default files first
df_vehicle = pd.concat([df_vehicle_1, df_vehicle_2], ignore_index=True)


# Standardize column names

def standardize_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = (
        df.columns.str.strip()
                  .str.lower()
                  .str.replace(r"[^a-z0-9]+", "_", regex=True)
                  .str.strip("_")
    )
    return df

df_auto = standardize_columns(df_auto)
df_vehicle = standardize_columns(df_vehicle)

# Column mapping dictionaries.
# These map likely source-specific names to a common target schema.
auto_map = {
    "customer_income": "income",
    "annual_income": "income",
    "applicant_income": "income",
    "loan_status": "default_flag",
    "default": "default_flag",
    "interest": "interest_rate",
    "loan_tenure_months": "term",
    "loan_term": "term",
    "employment_status": "employment_type",
    "loan_purpose": "purpose"
}

vehicle_map = {
    "disbursed_amount": "loan_amount",
    "asset_cost": "vehicle_cost",
    "ltv": "loan_to_value",
    "employment_type": "employment_type",
    "perform_cns_score": "credit_score",
    "pri_no_of_accts": "prior_accounts_count",
    "pri_overdue_accts": "prior_overdue_accounts",
    "primary_instal_amt": "monthly_payment_est",
    "loan_default": "default_flag",
    "state_id": "state_id",
    "branch_id": "branch_id",
    "manufacturer_id": "manufacturer_id",
    "employee_code_id": "employee_code_id"
}

# Apply only mappings for columns that exist
df_auto = df_auto.rename(columns={k: v for k, v in auto_map.items() if k in df_auto.columns})
df_vehicle = df_vehicle.rename(columns={k: v for k, v in vehicle_map.items() if k in df_vehicle.columns})

print("Columns after standardization and renaming:")
print("  Auto dataset sample columns:", list(df_auto.columns[:15]))
print("  Vehicle dataset sample columns:", list(df_vehicle.columns[:15]))


In [ ]:
# Align data types
def coerce_numeric_if_present(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df

numeric_candidates = [
    "loan_amount", "interest_rate", "term", "income", "credit_score",
    "loan_to_value", "vehicle_cost", "monthly_payment_est", "default_flag",
    "prior_accounts_count", "prior_overdue_accounts", "branch_id",
    "state_id", "manufacturer_id", "employee_code_id"
]

df_auto = coerce_numeric_if_present(df_auto, numeric_candidates)
df_vehicle = coerce_numeric_if_present(df_vehicle, numeric_candidates)

# Try to normalize default_flag into 0/1 if it exists
def normalize_default_flag(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "default_flag" in df.columns:
        if df["default_flag"].dtype == object:
            mapping = {
                "yes": 1, "y": 1, "default": 1, "charged_off": 1, "1": 1,
                "no": 0, "n": 0, "paid": 0, "current": 0, "0": 0
            }
            df["default_flag"] = (
                df["default_flag"].astype(str).str.strip().str.lower().map(mapping)
            )
        df["default_flag"] = pd.to_numeric(df["default_flag"], errors="coerce")
    return df

df_auto = normalize_default_flag(df_auto)
df_vehicle = normalize_default_flag(df_vehicle)

# Harmonize schemas

all_cols = sorted(set(df_auto.columns).union(set(df_vehicle.columns)))

df_auto = df_auto.reindex(columns=all_cols)
df_vehicle = df_vehicle.reindex(columns=all_cols)


# Add dataset source flag
df_auto["source"] = "auto_loan_dataset"
df_vehicle["source"] = "vehicle_loan_default_dataset"

print("Schema aligned.")
print(f"  Auto aligned shape: {df_auto.shape}")
print(f"  Vehicle aligned shape: {df_vehicle.shape}")


In [ ]:
# Concatenate the standardized datasets
df_combined = pd.concat([df_auto, df_vehicle], ignore_index=True)

print("Combined dataset shape before missing-value handling and synthetic expansion:")
print(df_combined.shape)


# Handle missing values
numeric_cols = df_combined.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df_combined.select_dtypes(exclude=[np.number]).columns.tolist()

for col in numeric_cols:
    if df_combined[col].isna().any():
        df_combined[col] = df_combined[col].fillna(df_combined[col].median())

for col in categorical_cols:
    if df_combined[col].isna().any():
        mode_val = df_combined[col].mode(dropna=True)
        fill_val = mode_val.iloc[0] if not mode_val.empty else "Unknown"
        df_combined[col] = df_combined[col].fillna(fill_val)

# Add a few engineered fields if enough source columns exist
if "loan_amount" in df_combined.columns and "income" in df_combined.columns:
    safe_income = df_combined["income"].replace(0, np.nan)
    df_combined["loan_to_income"] = df_combined["loan_amount"] / safe_income
    df_combined["loan_to_income"] = df_combined["loan_to_income"].replace([np.inf, -np.inf], np.nan).fillna(0)

if "loan_amount" in df_combined.columns and "term" in df_combined.columns:
    safe_term = df_combined["term"].replace(0, np.nan)
    df_combined["monthly_payment_est"] = (df_combined["loan_amount"] / safe_term).replace([np.inf, -np.inf], np.nan).fillna(0)

# Synthetic data expansion to exceed 500,000 rows
# Uses bootstrap sampling plus light noise on numeric columns
TARGET_ROWS = 500_001

def expand_with_synthetic_rows(df: pd.DataFrame, target_rows: int, random_state: int = 42) -> pd.DataFrame:
    df = df.copy()
    current_rows = len(df)
    if current_rows >= target_rows:
        df["record_origin"] = df.get("record_origin", "original")
        return df

    rows_needed = target_rows - current_rows
    synthetic = df.sample(n=rows_needed, replace=True, random_state=random_state).copy()

    rng = np.random.default_rng(random_state)
    synthetic["record_origin"] = "synthetic_bootstrap"

    numeric_cols_local = synthetic.select_dtypes(include=[np.number]).columns.tolist()
    protected_cols = {"default_flag", "branch_id", "state_id", "manufacturer_id", "employee_code_id"}

    for col in numeric_cols_local:
        if col in protected_cols:
            continue

        std_val = synthetic[col].std()
        if pd.notna(std_val) and std_val > 0:
            noise = rng.normal(loc=0.0, scale=0.01 * std_val, size=len(synthetic))
            synthetic[col] = synthetic[col] + noise

            # keep inherently non-negative financial fields non-negative
            if any(token in col for token in ["loan", "income", "amount", "rate", "score", "cost", "value", "payment", "term"]):
                synthetic[col] = synthetic[col].clip(lower=0)

    df["record_origin"] = "original"
    combined = pd.concat([df, synthetic], ignore_index=True)
    return combined

df_final = expand_with_synthetic_rows(df_combined, TARGET_ROWS, random_state=42)

print("Final dataset shape:")
print(df_final.shape)
print("\nSource counts:")
print(df_final["source"].value_counts())
print("\nRecord origin counts:")
print(df_final["record_origin"].value_counts())

output_file = "auto_finance_combined_over_500k.csv"
df_final.to_csv(output_file, index=False)

print(f"\nSaved final file as: {output_file}")


In [ ]:

# Review the final dataset
print(df_final.info())
df_final.head()


## Creative twist: Portfolio Risk Pulse Check

This section adds a simple interactive quiz you can mention in the technical report as the creative twist. It for the analysis of a short decision-support check rather than a passive display.

In [ ]:
try:
    import ipywidgets as widgets
    from IPython.display import Markdown, clear_output

    # Note: `final_df` and other variables like `borrower_segment` are not defined in the current kernel state.
    # For demonstration, I will use placeholder values or derive from `df_final` if possible.
    # Assuming `df_final` is the combined dataframe from previous steps.
    if 'loan_to_income' in df_final.columns:
        median_lti = round(df_final['loan_to_income'].median(), 3)
    else:
        median_lti = 0.0 # Placeholder

    # Use 'default' column if 'default_flag' is empty
    if 'default_flag' in df_final.columns and not df_final['default_flag'].isnull().all():
        default_rate = round(df_final['default_flag'].mean() * 100, 2)
    elif 'default' in df_final.columns and not df_final['default'].isnull().all():
        default_rate = round(df_final['default'].mean() * 100, 2)
    else:
        default_rate = 0.0 # Placeholder

    # `borrower_segment` is not in `df_final` columns; using a placeholder.
    high_income_label = "High Income Segment" # Placeholder

    q1 = widgets.RadioButtons(
        options=['loan_to_income_ratio', 'monthly_payment', 'source'],
        description='Q1'
    )
    q2 = widgets.RadioButtons(
        options=[f'{default_rate}%', f'{default_rate + 10:.2f}%', f'{max(default_rate - 5, 0):.2f}%'],
        description='Q2'
    )
    q3 = widgets.RadioButtons(
        options=['The storyboard narrative', 'The CSV file size', 'The notebook theme'],
        description='Q3'
    )

    button = widgets.Button(description='Check Answers', button_style='success')
    output = widgets.Output()

    def on_click(_):
        with output:
            clear_output()
            score = 0
            if q1.value == 'loan_to_income_ratio':
                score += 1
            if q2.value == f'{default_rate}%':
                score += 1
            if q3.value == 'The storyboard narrative':
                score += 1

            print(f'Your score: {score}/3')
            print(f'- Median loan-to-income ratio in this dataset: {median_lti}')
            print(f'- Default rate in this dataset: {default_rate}%')
            print('- The creative twist helps reinforce interpretation and storytelling.')

    button.on_click(on_click)

    display(Markdown('**Portfolio Risk Pulse Check**'))
    display(Markdown('1. Which engineered feature is central to the box plot comparison?'))
    display(q1)
    display(Markdown('2. What is the approximate default rate in the combined dataset?'))
    display(q2)
    display(Markdown('3. Which item most directly supports storyboarding in the technical report?'))
    display(q3)
    display(button, output)

except Exception as e:
    print('ipywidgets not available in this environment. You can still mention the quiz concept in the report.')
    print('Error:', e)

## Summary statistics for report support

In [ ]:
summary_table = pd.DataFrame({
    'metric': ['row_count', 'column_count', 'default_rate_pct', 'median_borrower_income', 'median_loan_amount', 'median_loan_to_income_ratio'],
    'value': [
        len(df_final),
        df_final.shape[1],
        round(df_final['default'].mean() * 100, 2), # Changed from 'default_flag' to 'default'
        round(df_final['income'].median(), 2),
        round(df_final['loan_amount'].median(), 2),
        round(df_final['loan_to_income'].median(), 3)
    ]
})
summary_table

## Visualizations

In [ ]:
import plotly.express as px

# Histogram: loan amount distribution (Plotly)
sample_hist = df_final.sample(n=min(100000, len(df_final)), random_state=42)

fig_loan_hist = px.histogram(
    sample_hist,
    x='loan_amount',
    nbins=50,
    title='Distribution of Loan Amounts',
    marginal='box',
    opacity=0.8
)
fig_loan_hist.update_layout(xaxis_title='Loan Amount', yaxis_title='Count')
fig_loan_hist.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram: borrower income distribution (Seaborn)
plt.figure(figsize=(11, 6))
sns.histplot(sample_hist['income'], bins=50, kde=True)
plt.title('Distribution of Borrower Income')
plt.xlabel('Borrower Income')
plt.ylabel('Count')
plt.tight_layout()
plt.show()

* There is a strong concentration around approximately $10,000.

In [ ]:
# Box Plot: loan-to-income ratio across borrower segments

# Create borrower_segment in sample_hist based on median income
# Using median_borrower_income from summary_table (derived from df_final) for consistency
median_income_val = 60000.0 # From summary_table['value'][summary_table['metric'] == 'median_borrower_income'].iloc[0]

sample_hist['borrower_segment'] = sample_hist['income'].apply(lambda x: 'High Income' if x >= median_income_val else 'Low Income')

plt.figure(figsize=(11, 6))
sns.boxplot(data=sample_hist, x='borrower_segment', y='loan_to_income')
plt.title('Loan-to-Income Ratio Across Borrower Segments')
plt.xlabel('Borrower Segment')
plt.ylabel('Loan-to-Income Ratio')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()

* Both borrow segments have similar baseline, ~16%.
* Low-income borrowers are more likely to carry higher loan burdens relative to income.

In [ ]:
# Scatter Plot: loan amount vs borrower income colored by default risk
sample_scatter = df_final.sample(n=min(30000, len(df_final)), random_state=42)

fig_scatter = px.scatter(
    sample_scatter,
    x='income',
    y='loan_amount',
    color='default',
    opacity=0.45,
    title='Loan Amount vs Borrower Income by Default Risk',
    hover_data=['loan_to_income', 'source', 'record_origin']
)
fig_scatter.update_layout(xaxis_title='Borrower Income', yaxis_title='Loan Amount')
fig_scatter.show()

* Most borrowers are concentrated within lower to mid-income ranges, while loan amounts vary significantly within those same income levels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Correlation Heatmap: financial and risk variables
corr_cols = ['income', 'loan_amount', 'int_rate', 'monthly_payment_est', 'loan_to_income', 'default']
corr_cols = [c for c in corr_cols if c in df_final.columns]

corr_matrix = df_final[corr_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr_matrix, annot=True, cmap='YlGnBu', fmt='.2f', square=False)
plt.title('Correlation Heatmap of Financial and Risk Variables')
plt.tight_layout()
plt.show()

*  There is no high correlation with any combination of varirables.  The loan_to_income and loan_amount have the highest correlation of 62%.

### Visualization Comments

Use the following interpretation prompts in your report:
* **Histogram:** Describe the concentration and spread of loan amounts and borrower income. Note skewness where applicable.
* **Box plot:** Discuss whether lower-income borrower segments show wider or higher loan-to-income ratios than higher-income segments.
* **Scatter plot:** Explain where default observations cluster and whether default risk appears more concentrated in certain income or loan ranges.
* **Heatmap:** Highlight the strongest positive and negative numeric relationships, especially those tied to "default_flag" and "loan_to_income_ratio".

# Part 2: Advanced Visualization, Mixed Media, and Segmented Reveals

This section extends the Part 1 notebook by adding the Part 2 requirements:

1. Four or more high-quality Plotly visualizations that progressively reveal insights.
2. Advanced design elements, including a consistent colorblind-friendly palette, annotations, and guide text.
3. Small multiples, a slope graph, and layered charts for multi-faceted comparison.
4. Mixed media and segmented reveals to support the final technical report and future screencast.


In [ ]:
# Part 2 setup: create a clean visualization dataset from df_final

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import HTML, display, Markdown

try:
    import google.colab  # type: ignore
    pio.renderers.default = "colab"
except Exception:
    pio.renderers.default = "notebook_connected"

# Colorblind-friendly Okabe-Ito-inspired palette
COLOR_SEQUENCE = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442"]
DEFAULT_COLOR_MAP = {0: "#0072B2", 1: "#D55E00", "0": "#0072B2", "1": "#D55E00"}

if "df_final" not in globals():
    raise NameError("df_final was not found. Run the data preparation cells above before running Part 2.")

viz_df = df_final.copy()

# Ensure required numeric fields are numeric
for col in ["income", "loan_amount", "int_rate", "monthly_payment_est", "loan_to_income", "default"]:
    if col in viz_df.columns:
        viz_df[col] = pd.to_numeric(viz_df[col], errors="coerce")

# Create fields if missing
if "loan_to_income" not in viz_df.columns and {"loan_amount", "income"}.issubset(viz_df.columns):
    viz_df["loan_to_income"] = viz_df["loan_amount"] / viz_df["income"].replace(0, np.nan)

if "default" not in viz_df.columns:
    viz_df["default"] = np.nan

if "source" not in viz_df.columns:
    viz_df["source"] = "combined_dataset"

# Clean impossible or unusable values for visualization
viz_df = viz_df.replace([np.inf, -np.inf], np.nan)
viz_df = viz_df.dropna(subset=["income", "loan_amount"])
viz_df = viz_df[(viz_df["income"] > 0) & (viz_df["loan_amount"] > 0)]

# Cap extreme values for readability while preserving the analytical dataset
income_cap = viz_df["income"].quantile(0.99)
loan_cap = viz_df["loan_amount"].quantile(0.99)
lti_cap = viz_df["loan_to_income"].quantile(0.99) if "loan_to_income" in viz_df.columns else np.nan

viz_df["income_capped"] = viz_df["income"].clip(upper=income_cap)
viz_df["loan_amount_capped"] = viz_df["loan_amount"].clip(upper=loan_cap)
viz_df["loan_to_income_capped"] = viz_df["loan_to_income"].clip(upper=lti_cap)

income_median = viz_df["income"].median()
viz_df["borrower_segment"] = np.where(viz_df["income"] >= income_median, "High Income", "Low Income")
viz_df["borrower_segment"] = pd.Categorical(viz_df["borrower_segment"], categories=["Low Income", "High Income"], ordered=True)

# Risk bands based on loan-to-income burden
viz_df["risk_band"] = pd.cut(
    viz_df["loan_to_income_capped"],
    bins=[-np.inf, 0.20, 0.40, np.inf],
    labels=["Lower Burden", "Moderate Burden", "Higher Burden"]
)

viz_df["default_label"] = viz_df["default"].fillna(0).round().astype(int).astype(str)
viz_df["source_label"] = viz_df["source"].astype(str).str.replace("_", " ").str.title()

# Sampling keeps interactive charts responsive in Colab
plot_df = viz_df.sample(n=min(100000, len(viz_df)), random_state=42).copy()
scatter_df = viz_df.sample(n=min(30000, len(viz_df)), random_state=42).copy()

print("Visualization rows available:", len(viz_df))
print("Rows used for main interactive plots:", len(plot_df))
print("Income cap for charts:", round(income_cap, 2))
print("Loan amount cap for charts:", round(loan_cap, 2))


## Segmented Reveal 1 — Portfolio Shape

**Story purpose:** Start broad by showing where the portfolio is concentrated before moving into borrower risk.  
**Design concept for the technical report:** A histogram with a marginal box plot is used to show both volume concentration and spread. The x-axis is capped at the 99th percentile so a small number of extreme outliers do not compress the main story.


In [ ]:
# Visualization 1: Layered histogram with marginal box plot
fig1 = px.histogram(
    plot_df,
    x="loan_amount_capped",
    color="borrower_segment",
    marginal="box",
    nbins=60,
    opacity=0.75,
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Segmented Reveal 1: Portfolio Shape — Loan Amount Distribution by Borrower Segment",
    labels={
        "loan_amount_capped": "Loan Amount (99th Percentile Cap)",
        "count": "Record Count",
        "borrower_segment": "Borrower Segment"
    }
)

fig1.add_vline(
    x=plot_df["loan_amount_capped"].median(),
    line_dash="dash",
    line_color="#333333",
    annotation_text="Median loan amount",
    annotation_position="top right"
)

fig1.add_annotation(
    x=0.02, y=0.95, xref="paper", yref="paper",
    text="Reveal 1: Identify the portfolio's center before evaluating risk.",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="#333333",
    align="left"
)

fig1.update_layout(
    template="plotly_white",
    bargap=0.05,
    legend_title_text="Borrower Segment",
    font=dict(size=12),
    title_font=dict(size=18)
)

fig1.show()
fig1.write_html("part2_visual_1_portfolio_shape.html")


## Segmented Reveal 2 — Risk Concentration by Segment

**Story purpose:** Move from portfolio shape to risk burden by comparing loan-to-income ratios across borrower segments.  
**Design concept for the technical report:** A box plot is used because averages alone can hide tail risk. Outliers are intentionally shown to make the risk concentration visible.


In [ ]:
# Visualization 2: Box plot with annotation overlay
fig2 = px.box(
    plot_df,
    x="borrower_segment",
    y="loan_to_income_capped",
    color="borrower_segment",
    points="outliers",
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Segmented Reveal 2: Risk Concentration — Loan-to-Income Ratio by Segment",
    labels={
        "borrower_segment": "Borrower Segment",
        "loan_to_income_capped": "Loan-to-Income Ratio (99th Percentile Cap)"
    }
)

fig2.add_hline(
    y=0.40,
    line_dash="dash",
    line_color="#D55E00",
    annotation_text="Elevated burden reference line: 0.40",
    annotation_position="top left"
)

fig2.add_annotation(
    x=0.03, y=0.92, xref="paper", yref="paper",
    text="Reveal 2: The center may look stable, but outliers show where risk concentrates.",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="#333333",
    align="left"
)

fig2.update_layout(
    template="plotly_white",
    showlegend=False,
    font=dict(size=12),
    title_font=dict(size=18)
)

fig2.show()
fig2.write_html("part2_visual_2_risk_concentration.html")


## Segmented Reveal 3 — Small Multiples for Default and Risk Burden

**Story purpose:** Compare default behavior across borrower segments and risk bands without forcing all information into one crowded chart.  
**Design concept for the technical report:** Small multiples separate high-income and low-income borrowers, making the comparison easier and more accessible.


In [ ]:
# Visualization 3: Small multiples showing default rate by risk band and borrower segment
small_mult = (
    viz_df.dropna(subset=["risk_band", "default"])
    .groupby(["borrower_segment", "risk_band"], observed=True)
    .agg(
        default_rate=("default", "mean"),
        record_count=("default", "size"),
        avg_loan_amount=("loan_amount", "mean"),
        avg_lti=("loan_to_income", "mean")
    )
    .reset_index()
)

small_mult["default_rate_pct"] = small_mult["default_rate"] * 100

fig3 = px.bar(
    small_mult,
    x="risk_band",
    y="default_rate_pct",
    color="risk_band",
    facet_col="borrower_segment",
    text=small_mult["default_rate_pct"].round(2).astype(str) + "%",
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Segmented Reveal 3: Small Multiples — Default Rate by Risk Band and Borrower Segment",
    labels={
        "risk_band": "Loan-to-Income Risk Band",
        "default_rate_pct": "Default Rate (%)",
        "borrower_segment": "Borrower Segment"
    },
    hover_data=["record_count", "avg_loan_amount", "avg_lti"]
)

fig3.add_annotation(
    x=0.02, y=1.10, xref="paper", yref="paper",
    text="Reveal 3: Compare segments side-by-side before drawing conclusions about risk.",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="#333333",
    align="left"
)

fig3.update_traces(textposition="outside")
fig3.update_layout(
    template="plotly_white",
    legend_title_text="Risk Band",
    font=dict(size=12),
    title_font=dict(size=18)
)

fig3.show()
fig3.write_html("part2_visual_3_small_multiples_default_rate.html")


## Segmented Reveal 4 — Slope Graph of Financial Burden

**Story purpose:** Show how average loan-to-income burden changes from low-income to high-income borrowers across data sources.  
**Design concept for the technical report:** A slope graph focuses attention on direction and difference instead of overwhelming the viewer with raw points.


In [ ]:
# Visualization 4: Slope graph comparing average LTI from low-income to high-income segments by source
slope_df = (
    viz_df.dropna(subset=["loan_to_income", "borrower_segment"])
    .groupby(["source_label", "borrower_segment"], observed=True)
    .agg(avg_lti=("loan_to_income", "mean"), records=("loan_to_income", "size"))
    .reset_index()
)

fig4 = px.line(
    slope_df,
    x="borrower_segment",
    y="avg_lti",
    color="source_label",
    markers=True,
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Segmented Reveal 4: Slope Graph — Average Loan-to-Income Burden by Source and Segment",
    labels={
        "borrower_segment": "Borrower Segment",
        "avg_lti": "Average Loan-to-Income Ratio",
        "source_label": "Data Source"
    },
    hover_data=["records"]
)

fig4.add_annotation(
    x=0.03, y=0.95, xref="paper", yref="paper",
    text="Reveal 4: The slope shows whether financial burden changes across segments.",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="#333333",
    align="left"
)

fig4.update_layout(
    template="plotly_white",
    font=dict(size=12),
    title_font=dict(size=18),
    legend_title_text="Data Source"
)

fig4.show()
fig4.write_html("part2_visual_4_slope_graph_lti.html")


## Segmented Reveal 5 — Layered Borrower Profile

**Story purpose:** Bring income, loan amount, loan burden, and default status together in one layered view.  
**Design concept for the technical report:** The scatter plot uses color for risk burden and symbol for default status. A reference line is added to show where loan amount equals 40% of income, helping the audience interpret financial burden visually.


In [ ]:
# Visualization 5: Layered scatter plot with reference line
fig5 = px.scatter(
    scatter_df,
    x="income_capped",
    y="loan_amount_capped",
    color="risk_band",
    symbol="default_label",
    opacity=0.55,
    color_discrete_sequence=COLOR_SEQUENCE,
    title="Segmented Reveal 5: Layered Borrower Profile — Income, Loan Amount, Risk Band, and Default",
    labels={
        "income_capped": "Borrower Income (99th Percentile Cap)",
        "loan_amount_capped": "Loan Amount (99th Percentile Cap)",
        "risk_band": "Loan-to-Income Risk Band",
        "default_label": "Default Flag"
    },
    hover_data=["borrower_segment", "loan_to_income_capped", "source_label"]
)

x_line = np.linspace(0, scatter_df["income_capped"].max(), 100)
fig5.add_trace(
    go.Scatter(
        x=x_line,
        y=0.40 * x_line,
        mode="lines",
        name="0.40 loan-to-income reference",
        line=dict(color="#333333", dash="dash")
    )
)

fig5.add_annotation(
    x=0.02, y=0.95, xref="paper", yref="paper",
    text="Reveal 5: Layered view connects borrower capacity, loan size, burden, and outcome.",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="#333333",
    align="left"
)

fig5.update_layout(
    template="plotly_white",
    font=dict(size=12),
    title_font=dict(size=18),
    legend_title_text="Risk Story"
)

fig5.show()
fig5.write_html("part2_visual_5_layered_borrower_profile.html")


## Segmented Reveal 6 — Relationship Validation

**Story purpose:** Close the visual sequence by validating which variables move together and which variables do not.  
**Design concept for the technical report:** A correlation heatmap is used as the final reveal because it summarizes relationships after the audience has already seen the individual visual evidence.


In [ ]:
# Visualization 6: Plotly correlation heatmap
corr_cols = ["income", "loan_amount", "int_rate", "monthly_payment_est", "loan_to_income", "default"]
corr_cols = [c for c in corr_cols if c in viz_df.columns]
corr_df = viz_df[corr_cols].replace([np.inf, -np.inf], np.nan).dropna()
corr_matrix = corr_df.corr(numeric_only=True).round(2)

fig6 = px.imshow(
    corr_matrix,
    text_auto=True,
    color_continuous_scale="RdBu",
    zmin=-1,
    zmax=1,
    title="Segmented Reveal 6: Relationship Validation — Correlation Heatmap of Financial and Risk Variables",
    labels=dict(color="Correlation")
)

fig6.add_annotation(
    x=0.02, y=1.12, xref="paper", yref="paper",
    text="Reveal 6: Use correlation to validate patterns, not as the only source of truth.",
    showarrow=False,
    bgcolor="rgba(255,255,255,0.85)",
    bordercolor="#333333",
    align="left"
)

fig6.update_layout(
    template="plotly_white",
    font=dict(size=12),
    title_font=dict(size=18)
)

fig6.show()
fig6.write_html("part2_visual_6_correlation_heatmap.html")


# Mixed Media and Segmented Reveal Support

This section adds non-chart storytelling elements that can be referenced in the technical report and reused in the Part 3 screencast. The goal is to guide the audience through the story step-by-step instead of showing all insights at once.


In [ ]:
# Mixed media element: storyboard cards + narration script for screencast
storyboard_html = '''
<div style="font-family: Arial, sans-serif; line-height:1.45; max-width: 980px;">
  <h2 style="color:#1f4e79;">Automotive Finance Portfolio Risk: Segmented Reveal Storyboard</h2>
  <p><strong>Audience:</strong> Credit risk, portfolio analytics, and business stakeholders.</p>
  <p><strong>Purpose:</strong> Move from broad portfolio structure to specific credit-risk interpretation.</p>

  <div style="display:grid; grid-template-columns: repeat(2, minmax(280px, 1fr)); gap:14px;">
    <div style="border-left:6px solid #0072B2; padding:12px; background:#f7fbff;">
      <h3>Reveal 1: Portfolio Shape</h3>
      <p>Start with the distribution of loan amounts to identify where the portfolio is concentrated.</p>
    </div>
    <div style="border-left:6px solid #E69F00; padding:12px; background:#fffaf0;">
      <h3>Reveal 2: Risk Concentration</h3>
      <p>Use loan-to-income ratio to move from volume to borrower burden and tail risk.</p>
    </div>
    <div style="border-left:6px solid #009E73; padding:12px; background:#f3fff9;">
      <h3>Reveal 3: Segment Comparison</h3>
      <p>Use small multiples to compare default behavior across borrower segments and risk bands.</p>
    </div>
    <div style="border-left:6px solid #CC79A7; padding:12px; background:#fff7fc;">
      <h3>Reveal 4: Directional Burden</h3>
      <p>Use the slope graph to show how average financial burden shifts between income groups.</p>
    </div>
    <div style="border-left:6px solid #56B4E9; padding:12px; background:#f5fcff;">
      <h3>Reveal 5: Layered Profile</h3>
      <p>Combine income, loan amount, default, and risk band in a layered borrower-level view.</p>
    </div>
    <div style="border-left:6px solid #D55E00; padding:12px; background:#fff8f3;">
      <h3>Reveal 6: Validate Relationships</h3>
      <p>Close with the heatmap to validate patterns and identify model limitations.</p>
    </div>
  </div>

  <h3 style="margin-top:20px;">Narration Script for Screencast</h3>
  <p>
    When I walk through this analysis, I begin with the portfolio shape because it shows where most loans are concentrated.
    Then I move into loan-to-income burden, which is more meaningful for credit risk than loan amount alone.
    The small multiples and slope graph allow me to compare borrower segments without overwhelming the viewer.
    Finally, I use the layered scatter plot and heatmap to connect individual borrower patterns back to overall relationships.
    This segmented reveal process helps the audience build understanding step by step.
  </p>
</div>
'''

display(HTML(storyboard_html))


In [ ]:
# Optional mixed media: browser-based narration button
# Note: This uses the browser's built-in speech synthesis. It may work differently depending on browser and Colab settings.

narration_text = (
    "This segmented reveal begins with portfolio shape, then moves to borrower burden, "
    "segment comparison, directional risk, layered borrower profiles, and relationship validation. "
    "The purpose is to guide the audience from descriptive analysis to credit risk interpretation."
)

display(HTML(f'''
<div style="font-family: Arial, sans-serif; padding:14px; border:1px solid #cccccc; border-radius:8px; max-width:900px;">
  <h3>Optional Audio-Style Narration</h3>
  <p>This button uses browser speech synthesis as a mixed-media storytelling element.</p>
  <button onclick="speechSynthesis.speak(new SpeechSynthesisUtterance(`{narration_text}`));"
          style="padding:10px 14px; border-radius:6px; border:0; background:#0072B2; color:white; cursor:pointer;">
    Play Narrative Summary
  </button>
</div>
'''))


## Technical Report Justification Notes

Use the following notes in the Part 2 technical report:

- **Consistent palette:** The notebook uses a colorblind-friendly palette to improve accessibility and reduce confusion across visuals.
- **Segmented reveals:** Each chart builds on the previous chart, moving from portfolio distribution to borrower burden, segment comparison, layered borrower profiles, and relationship validation.
- **Annotated overlays:** Reference lines and text callouts guide the audience toward the intended interpretation.
- **Small multiples:** Used to compare default rates across borrower segments without overcrowding the chart.
- **Slope graph:** Used to show the directional shift in average loan-to-income burden across income segments and source datasets.
- **Layered chart:** Used to combine income, loan amount, risk band, and default flag in one view while still preserving interpretability.
- **Mixed media:** Storyboard cards and optional browser narration support the future screencast and help connect visual analysis to audience understanding.
